In [1]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns
import sqlite3
import yaml

from _configs.country_config import *
from _configs.files_config import *
from _configs.run_config import *

from _utils.utils import *

In [3]:
# if validation path ends in a number new_validation_path = validation_path but increment the number. else new_validation_path = validation_path + "_1"
if validation_path.endswith(tuple("0123456789")):
    # Extract the numeric suffix from the validation_path
    suffix = ""
    for char in reversed(validation_path):
        if char.isdigit():
            suffix = char + suffix
        else:
            break

    # Increment the numeric suffix
    new_suffix = str(int(suffix) + 1)

    # Create the new validation path by replacing the old suffix with the new one
    new_validation_path = validation_path[:-len(suffix)] + new_suffix
else:
    new_validation_path = validation_path + "_1"

print(new_validation_path)

os.makedirs(new_validation_path, exist_ok=True)


input_data = yaml.safe_load(Path(f"{VALIDATION_RUN_INPUTS_DIR}/input_validation.yml").read_text(encoding="utf-8"))
population_scale = input_data["population_demographic"]["artificial_rescaling_of_population_size"]
starting_date = input_data["simulation_timeframe"]["starting_date"]
birth_Rate = input_data["population_demographic"]["birth_rate"]

# 1) Use the original dict
country_dict = {
    "population_scale": population_scale,
    "birth_rate": birth_Rate,
    }
country_json = json.dumps(country_dict, ensure_ascii=False)
# 3) Attribute-style access (SimpleNamespace)
from types import SimpleNamespace
country = json.loads(country_json, object_hook=lambda d: SimpleNamespace(**d))
# print(country.country_code)

exp_path = validation_path
output_path = Path(exp_path) / "output"
analysis_path = Path(exp_path) / "analysis"
os.makedirs(analysis_path, exist_ok=True)

print(data_path)
print(output_path)
print(analysis_path)

validation_runs/validation_init_pop_21.12M.asc_0.25_population_scale_multiple_pattern_20_replicates_1
DATA
validation_runs/validation_init_pop_21.12M.asc_0.25_population_scale_multiple_pattern_20_replicates/output
validation_runs/validation_init_pop_21.12M.asc_0.25_population_scale_multiple_pattern_20_replicates/analysis


In [4]:
# =============================================================================
# Scale district beta to minimize gap between observed and simulated cases
# Creates a new beta raster instead of changing seasonality
# Uses read_raster / write_raster helpers
# =============================================================================

# ── 0. Paths ─────────────────────────────────────────────────────────────────
beta_path = f"{VALIDATION_RUN_INPUTS_DIR}/{country_code}_beta.asc"
pop_path = projected_population_calibration_year_raster_path
district_path = districts_raster_path
compare_path = f"{analysis_path}/compare_data_pfpr_pop_incidence.csv"

new_beta_output_path = f"{new_validation_path}/{country_code}_beta_adjusted.asc"

# ── 2. Load rasters and compare table ────────────────────────────────────────
beta, beta_meta = read_raster(beta_path)
pop, pop_meta = read_raster(pop_path)
dist, dist_meta = read_raster(district_path)
compare = pd.read_csv(compare_path)

NODATA = beta_meta["NODATA_value"]

# Basic shape checks
if beta.shape != pop.shape or beta.shape != dist.shape:
    raise ValueError(
        f"Raster shapes do not match: beta={beta.shape}, pop={pop.shape}, dist={dist.shape}"
    )

# ── 3. Compute district ratios ───────────────────────────────────────────────
records = []

for _, row in compare.iterrows():
    uid = int(row["unit_id"])

    obs = float(row["annual_clinical_episodes_obs"])
    sim = float(row["mean_clinical_episodes_sim"])   # replace with mean_clinical_eps_sim if needed

    ratio = obs / sim if sim > 0 else 1.0

    mask = (
        (dist == uid) &
        (beta != NODATA) &
        (pop != NODATA)
    )

    pop_in_dist = pop[mask]
    beta_in_dist = beta[mask]

    total_pop = pop_in_dist.sum()
    wmean_beta = (
        (beta_in_dist * pop_in_dist).sum() / total_pop
        if total_pop > 0 else 0.0
    )

    records.append({
        "unit_id": uid,
        "district_name": row["district_name"],
        "obs_cases": obs,
        "sim_cases": sim,
        "ratio": ratio,
        "n_pixels": int(mask.sum()),
        "total_pop": total_pop,
        "old_wmean_beta": wmean_beta,
        "new_wmean_beta": wmean_beta * ratio,
    })

df = pd.DataFrame(records)

print(df[
    ["district_name", "obs_cases", "sim_cases", "ratio",
     "n_pixels", "total_pop", "old_wmean_beta", "new_wmean_beta"]
].to_string(index=False))

# ── 4. Create adjusted beta raster ───────────────────────────────────────────
beta_adj = beta.copy()

for _, r in df.iterrows():
    uid = int(r["unit_id"])
    ratio = float(r["ratio"])

    mask = (dist == uid) & (beta_adj != NODATA)
    beta_adj[mask] *= ratio

# Optional safety clamp
beta_adj[(beta_adj != NODATA) & (beta_adj < 0)] = 0.0

# ── 5. Save new beta raster ──────────────────────────────────────────────────
write_raster(
    raster=beta_adj,
    file=new_beta_output_path,
    xllcorner=beta_meta["xllcorner"],
    yllcorner=beta_meta["yllcorner"],
    cellsize=beta_meta["cellsize"],
    mask_raster=beta,   # preserve nodata footprint from original beta
    nodata=NODATA,
)

print(f"\nAdjusted beta raster saved to: {new_beta_output_path}")

# ── 6. Diagnostics ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# (a) Ratio by district
ax = axes[0, 0]
df_s = df.sort_values("ratio")
colors = ["#e74c3c" if r < 1 else "#2ecc71" for r in df_s["ratio"]]
ax.barh(df_s["district_name"], df_s["ratio"], color=colors, edgecolor="white")
ax.axvline(1.0, color="black", ls="--", lw=1)
for i, (_, row) in enumerate(df_s.iterrows()):
    ax.text(row["ratio"] + 0.02, i, f"{row['ratio']:.3f}", va="center", fontsize=9)
ax.set_xlabel("Ratio (obs / sim)")
ax.set_title("Beta adjustment ratio per district")

# (b) Old vs new weighted mean beta
ax = axes[0, 1]
y = np.arange(len(df))
df_b = df.sort_values("old_wmean_beta").reset_index(drop=True)
ax.barh(y - 0.2, df_b["old_wmean_beta"], height=0.35, label="Old β", color="#3498db")
ax.barh(y + 0.2, df_b["new_wmean_beta"], height=0.35, label="New β", color="#f39c12")
ax.set_yticks(y)
ax.set_yticklabels(df_b["district_name"])
ax.set_xlabel("Population-weighted mean beta")
ax.set_title("Old vs adjusted district β")
ax.legend()

# (c) Sim vs obs before, with arrows to target
ax = axes[1, 0]
maxv = max(df["obs_cases"].max(), df["sim_cases"].max()) * 1.15
ax.plot([0, maxv], [0, maxv], "r--", lw=1)
ax.scatter(df["obs_cases"], df["sim_cases"], s=100, c="#e74c3c", label="Before")
ax.scatter(df["obs_cases"], df["obs_cases"], s=100, c="#2ecc71", label="Target")
for _, r in df.iterrows():
    ax.annotate(
        r["district_name"],
        (r["obs_cases"], r["sim_cases"]),
        fontsize=7,
        xytext=(5, 5),
        textcoords="offset points",
    )
    ax.annotate(
        "",
        xy=(r["obs_cases"], r["obs_cases"]),
        xytext=(r["obs_cases"], r["sim_cases"]),
        arrowprops=dict(arrowstyle="->", color="green", lw=1.2, alpha=0.5),
    )
ax.set_xlabel("Observed cases")
ax.set_ylabel("Simulated cases")
ax.set_title("Sim vs Obs before adjustment")
ax.legend()
ax.set_xlim(0, maxv)
ax.set_ylim(0, maxv)

# (d) Histogram of beta values before vs after
ax = axes[1, 1]
valid_old = beta[beta != NODATA]
valid_new = beta_adj[beta_adj != NODATA]
ax.hist(valid_old, bins=50, alpha=0.6, label="Old beta")
ax.hist(valid_new, bins=50, alpha=0.6, label="Adjusted beta")
ax.set_xlabel("Beta")
ax.set_ylabel("Count")
ax.set_title("Distribution of beta values")
ax.legend()

plt.tight_layout()
plt.savefig(f"{analysis_path}/beta_adjustment_diagnostic.png", dpi=150, bbox_inches="tight")
plt.show()

# ── 7. Summary ───────────────────────────────────────────────────────────────
print("\n" + "=" * 90)
print("SUMMARY")
print("=" * 90)
print(f"{'District':<15} {'Obs':>12} {'Sim':>12} {'Ratio':>8} {'Old β':>12} {'New β':>12} {'Action'}")
print("-" * 90)

for _, r in df.sort_values("district_name").iterrows():
    action = "↓ scale down beta" if r["ratio"] < 1 else "↑ scale up beta"
    print(
        f"{r['district_name']:<15} "
        f"{r['obs_cases']:>12,.0f} "
        f"{r['sim_cases']:>12,.0f} "
        f"{r['ratio']:>8.3f} "
        f"{r['old_wmean_beta']:>12.6f} "
        f"{r['new_wmean_beta']:>12.6f} "
        f"{action}"
    )

KeyError: 'annual_clinical_episodes_obs'

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import TwoSlopeNorm

ASC_PATH = f"{data_path}/moz_districts.asc"
OLD_BETA_PATH = f"{validation_input_path}/moz_beta.asc"
NEW_BETA_PATH = f"{analysis_path}/moz_beta_adjusted.asc"

# =============================
# Convert nodata to NaN for plotting only
# =============================
def raster_to_nan(raster: np.ndarray, nodata: float) -> np.ndarray:
    out = raster.astype(float, copy=True)
    out[out == nodata] = np.nan
    return out


# =============================
# Load data
# =============================
district_grid_raw, district_meta = read_raster(ASC_PATH)
old_beta_raw, old_meta = read_raster(OLD_BETA_PATH)
new_beta_raw, new_meta = read_raster(NEW_BETA_PATH)

district_grid = raster_to_nan(district_grid_raw, district_meta["NODATA_value"])
old_beta = raster_to_nan(old_beta_raw, old_meta["NODATA_value"])
new_beta = raster_to_nan(new_beta_raw, new_meta["NODATA_value"])

beta_diff = new_beta - old_beta

# Mask outside district if wanted
valid_district = np.isfinite(district_grid)
old_beta_plot = np.where(valid_district, old_beta, np.nan)
new_beta_plot = np.where(valid_district, new_beta, np.nan)
beta_diff_plot = np.where(valid_district, beta_diff, np.nan)

# Shared color scale for old/new beta
beta_min = np.nanmin([np.nanmin(old_beta_plot), np.nanmin(new_beta_plot)])
beta_max = np.nanmax([np.nanmax(old_beta_plot), np.nanmax(new_beta_plot)])

# Symmetric color scale for difference
diff_abs = np.nanmax(np.abs(beta_diff_plot))
diff_norm = TwoSlopeNorm(vmin=-diff_abs, vcenter=0.0, vmax=diff_abs)

# =============================
# Plot
# =============================
fig, axes = plt.subplots(1, 3, figsize=(24, 8), constrained_layout=True)

# (1) Old beta
im0 = axes[0].imshow(old_beta_plot, cmap="viridis", vmin=beta_min, vmax=beta_max)
axes[0].set_title("Old Beta", fontsize=14)
axes[0].axis("off")
cbar0 = plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
cbar0.set_label("Beta")

# (2) New beta
im1 = axes[1].imshow(new_beta_plot, cmap="viridis", vmin=beta_min, vmax=beta_max)
axes[1].set_title("New Beta (Adjusted)", fontsize=14)
axes[1].axis("off")
cbar1 = plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
cbar1.set_label("Beta")

# (3) Difference
im2 = axes[2].imshow(beta_diff_plot, cmap="RdBu_r", norm=diff_norm)
axes[2].set_title("Beta Difference (New - Old)", fontsize=14)
axes[2].axis("off")
cbar2 = plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
cbar2.set_label("Delta Beta")

plt.savefig(f"{analysis_path}/beta_old_new_diff.png", dpi=200, bbox_inches="tight")
plt.show()

# =============================
# Summary stats
# =============================
print("Old beta:")
print(f"  min  = {np.nanmin(old_beta_plot):.6f}")
print(f"  max  = {np.nanmax(old_beta_plot):.6f}")
print(f"  mean = {np.nanmean(old_beta_plot):.6f}")

print("\nNew beta:")
print(f"  min  = {np.nanmin(new_beta_plot):.6f}")
print(f"  max  = {np.nanmax(new_beta_plot):.6f}")
print(f"  mean = {np.nanmean(new_beta_plot):.6f}")

print("\nDifference (new - old):")
print(f"  min  = {np.nanmin(beta_diff_plot):.6f}")
print(f"  max  = {np.nanmax(beta_diff_plot):.6f}")
print(f"  mean = {np.nanmean(beta_diff_plot):.6f}")

In [ ]:
OVERRIDE_INPUT_FOLDER = True
new_validation_log_path = new_validation_path / "log"
new_validation_output_path = new_validation_path / "output"
os.makedirs(new_validation_path, exist_ok=True)
os.makedirs(new_validation_log_path, exist_ok=True)
os.makedirs(new_validation_output_path, exist_ok=True)

# Copy bin,input,script folders from old validation to new validation folder
import shutil
shutil.copytree(f"{validation_path}/bin", new_validation_path / "bin", dirs_exist_ok=True)
if OVERRIDE_INPUT_FOLDER:
    print("Overriding input folder: copying input from old validation to new validation folder.")
    shutil.copytree(f"{validation_path}/input", new_validation_path / "input", dirs_exist_ok=True)
else:
    print("Not overriding input folder: skipping copy of input from old validation to new validation folder.")
shutil.copytree(f"{validation_path}/script", new_validation_path / "script", dirs_exist_ok=True)

# Copy new beta file to new validation input folder
shutil.copy(new_beta_output_path, new_validation_path / "input" / "moz_beta.asc")